In [101]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [102]:
# Preparing the dataset
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-10-11 14:45:27--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.16’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.005s  

2026-10-11 14:45:27 (49.7 MB/s) - ‘course_lead_scoring_2026.csv.16’ saved [278746/278746]



In [103]:
df = pd.read_csv('course_lead_scoring_2026.csv')

In [104]:
len(df)

5000

In [105]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [106]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [107]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)
numerical_columns = list(df.dtypes[df.dtypes != 'str'].index)
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [108]:
columns_with_missing_values = df.isna().any()
columns_with_missing_values.sum()

np.int64(6)

In [109]:
df[categorical_columns] = df[categorical_columns].fillna('NA')
df[numerical_columns] = df[numerical_columns].fillna(0.0)

## Question 1

In [110]:
mode_industry = df['industry'].mode()[0]
mode_industry

'technology'

## Question 2

In [111]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]
for col1, col2 in pairs:
    corr = df[[col1]].corrwith(df[col2]).iloc[0]
    print(f"{col1} - {col2}: {abs(corr):.4f}")

interaction_count - lead_score: 0.9157
number_of_courses_viewed - lead_score: 0.7572
number_of_courses_viewed - interaction_count: 0.7216
annual_income - interaction_count: 0.1228


In [112]:
# Split the data

In [113]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for col1, col2 in pairs:
    corr = df[[col1]].corrwith(df[col2]).iloc[0]
    print(f"{col1} - {col2}: {corr}")

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_full_train = df_full_train['converted'].values
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_full_train['converted']
del df_train['converted']
del df_val['converted']
del df_test['converted']

print(f"train: {len(df_train)}, val: {len(df_val)}, test: {len(df_test)}")

interaction_count - lead_score: 0.9157457980418688
number_of_courses_viewed - lead_score: 0.7572042392104885
number_of_courses_viewed - interaction_count: 0.7216090038937016
annual_income - interaction_count: 0.12284235135955969
train: 3000, val: 1000, test: 1000


## Question 3

In [114]:
def calculate_mi(series):
    return mutual_info_score(series, y_train)

df_mi = df_train[categorical_columns].apply(calculate_mi)
df_mi = df_mi.sort_values(ascending=False)

options = ['industry', 'location', 'lead_source', 'employment_status']

for opt in options:
    score = round(df_mi[opt], 2)
    print(f"{opt}: {score}")

industry: 0.0
location: 0.0
lead_source: 0.03
employment_status: 0.02


## Question 4

In [115]:
numerical_columns.remove('converted')
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)
accuracy_rounded = round(accuracy, 2)
accuracy_rounded

0.65

## Question 5

In [116]:
original_accuracy = accuracy_score(y_val, y_pred)
print(f"Original Accuracy: {original_accuracy:.4f}\n")

features_to_check = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
results = {}

for feat in features_to_check:
    # Create new features subsets excludinf the current one
    cat_subset = [c for c in categorical_columns if c != feat]
    num_subset = [n for n in numerical_columns if n != feat]
    
    # reaply DictVectorizer on the subset
    dv_sub = DictVectorizer(sparse=False)
    train_dict_sub = df_train[cat_subset + num_subset].to_dict(orient='records')
    X_train_sub = dv_sub.fit_transform(train_dict_sub)
    
    val_dict_sub = df_val[cat_subset + num_subset].to_dict(orient='records')
    X_val_sub = dv_sub.transform(val_dict_sub)
    
    # retain a model on the new subset
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    y_pred_sub = model_sub.predict(X_val_sub)
    
    # calculate the accuracy and the difference
    acc_sub = accuracy_score(y_val, y_pred_sub)
    diff = original_accuracy - acc_sub
    
    results[feat] = diff
    print(f"Without '{feat}': Acc = {acc_sub:.4f} | diff (Orig - New) = {diff:.4f}")

# smallest difference
least_useful_feature = min(results, key=lambda k: abs(results[k]))
least_useful_feature

Acuratețea originală: 0.6450

Without 'lead_source': Acc = 0.6420 | diff (Orig - New) = 0.0030
Without 'number_of_courses_viewed': Acc = 0.6430 | diff (Orig - New) = 0.0020
Without 'interaction_count': Acc = 0.6010 | diff (Orig - New) = 0.0440


'number_of_courses_viewed'

## Question 6

In [117]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
scores = {}
for c in c_values:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    acc_rounded = round(acc, 3)
    
    scores[c] = acc_rounded
    print(f"c = {c:<10} | Accuracy = {acc_rounded}")

best_c = max(scores, key=scores.get)
best_c

c = 1e-06      | Accuracy = 0.598
c = 1e-05      | Accuracy = 0.598
c = 0.0001     | Accuracy = 0.613
c = 0.001      | Accuracy = 0.645


0.001